In [ ]:
### RESULTS ARE IN: "dpa_tuning3_starting11122025_v4_data" ###

# Failed jobs:
# task_1441.status
# task_1472.status
# task_1505.status

In [1]:
import json
import subprocess
import re
from itertools import product
import xarray as xr
import pandas as pd
from pathlib import Path
import os
import numpy as np
import matplotlib.pyplot as plt

In [14]:
output_dir = "/work/fl53wumy-dpa_data/fl53wumy-llaae_data_new_22092025-1763346001/fl53wumy-llaae_data_new-1758244802/fl53wumy-llaae_data_new-1748049607/dpa_output/dpa_tuning5_starting13022026_v6_data/"
# 1) Define your coordinate values
coords = {
    'latent_dim':    [50, 100],
    'encoder':       ['learnable'],
    'hidden_dim_NN': [50, 100],
    'num_layers_NN': [4, 6],
    'noise_dim_dec': [5, 20, 100],
    'hidden_dim_lm': [50, 100],
    'noise_dim_lm':  [20, 100],
    'lamb':          [0.5, 1.0],
    'epoch':         np.arange(1, 101),          # 0 … 100 inclusive
    'batch_norm':    ["False", "True"],
    'alpha':         [1.0, 1.5],
    'learn_rate':    [0.0001, 0.00005],
    'mode':          ['train', 'test'],
    'loss':          ['Total loss','Total S1','Total S2',
                      'DPA NRGY','DPA s1','DPA s2',
                      'LM NRGY','LM S1','LM S2']
}


#save_dir = f"{settings['output_dir']}_{latent_dim}_{num_layers}_{hidden_dim}_{noise_dim_dec}_{in_dim_lm}_{noise_dim_lm}_{num_layers_lm}_{hidden_dim_lm}_encoderis{encoder}_lambda{lam}/"

In [15]:
# Generate all combinations as tuples
combos = list(product(
    coords['latent_dim'],
    coords['encoder'],
    coords['hidden_dim_NN'],
    coords['num_layers_NN'],
    coords['noise_dim_dec'],
    coords['hidden_dim_lm'],
    coords['noise_dim_lm'],
    coords['lamb'],
    coords['batch_norm'],
    coords['alpha'],
    coords['learn_rate']
))

# If you prefer a list of dicts:
param_names = [
    "latent_dim", "encoder", "hidden_dim_NN", "num_layers_NN",
    "noise_dim_dec", "hidden_dim_lm", "noise_dim_lm", "lambda", "batch_norm", "alpha", "learn_rate"
]
combo_dicts = [dict(zip(param_names, vals)) for vals in combos]

In [16]:
combo_dicts

[{'latent_dim': 50,
  'encoder': 'learnable',
  'hidden_dim_NN': 50,
  'num_layers_NN': 4,
  'noise_dim_dec': 5,
  'hidden_dim_lm': 50,
  'noise_dim_lm': 20,
  'lambda': 0.5,
  'batch_norm': 'False',
  'alpha': 1.0,
  'learn_rate': 0.0001},
 {'latent_dim': 50,
  'encoder': 'learnable',
  'hidden_dim_NN': 50,
  'num_layers_NN': 4,
  'noise_dim_dec': 5,
  'hidden_dim_lm': 50,
  'noise_dim_lm': 20,
  'lambda': 0.5,
  'batch_norm': 'False',
  'alpha': 1.0,
  'learn_rate': 5e-05},
 {'latent_dim': 50,
  'encoder': 'learnable',
  'hidden_dim_NN': 50,
  'num_layers_NN': 4,
  'noise_dim_dec': 5,
  'hidden_dim_lm': 50,
  'noise_dim_lm': 20,
  'lambda': 0.5,
  'batch_norm': 'False',
  'alpha': 1.5,
  'learn_rate': 0.0001},
 {'latent_dim': 50,
  'encoder': 'learnable',
  'hidden_dim_NN': 50,
  'num_layers_NN': 4,
  'noise_dim_dec': 5,
  'hidden_dim_lm': 50,
  'noise_dim_lm': 20,
  'lambda': 0.5,
  'batch_norm': 'False',
  'alpha': 1.5,
  'learn_rate': 5e-05},
 {'latent_dim': 50,
  'encoder': 'lear

In [17]:
def load_and_validate(path, validator):
    """
    Load the file at `path` and run `validator(obj)`.
    Raise FileNotFoundError if missing, or ValueError if validator fails.
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"No such file: {path}")
    
    # Example for JSON; swap out for pd.read_csv, pickle.load, etc.
    #with open(path, 'r') as f:
        #obj = json.load(f)
    df = pd.read_csv(
            path,
            skipinitialspace=True,
            dtype={'Epoch': str}
        )

    if not validator(df):
        raise ValueError(f"Validation failed for file: {path}")
    df_data = get_train_test(df)
    return df_data

def get_train_test(df):
    # split into Train vs Test
    is_train = df['Epoch'].str.startswith('Train')
    is_test  = df['Epoch'].str.startswith('Test')

    train_df = df[is_train].reset_index(drop=True)
    test_df  = df[is_test].reset_index(drop=True)

    # get numeric columns and arrays
    numeric_cols = df.columns.drop('Epoch')
    train_array = train_df[numeric_cols].to_numpy()
    test_array  = test_df[numeric_cols].to_numpy()
    return train_array, test_array

# --- Your custom validation function ---
def col_no(df):
    train_array, test_array = get_train_test(df)
    
    if test_array.shape[0] == 100:
        prop = True
    else:
        print(f"[{i}] Test array length: {test_array.shape[0]}")
        prop = False
    return prop

In [18]:
# 2) Create an empty Dataset filled with NaNs
ds = xr.Dataset(
    coords={k: ( [k], v ) for k, v in coords.items()},
    data_vars={
        'value': ( list(coords.keys()), 
                   np.full([len(v) for v in coords.values()], np.nan) )
    }
)
# dims of ds['value'] are in the order of coords.keys()

# 3) Function to parse hyperparameters from filename
def parse_filename(fname):
    """
    example filename:
    _10_4_50_10_1001_20_2_50_encoderislearnable_lambda0.0
    _latentdim(0)_numlayers(1)_hiddendim(2)_noisedimdec(3)_indimlm(4)_noisedimlm(5)_numlayerslm(6)_hiddendimlm(7)_encoder(8)_lambda(9)
    Expect filenames like:
      '..._10_PCA_50_4_5_20_20_20_0.5.txt'
    in exactly the same order as coords (except epoch/mode/loss).
    """
    stem = Path(fname).name
    print(stem)
    parts = stem.split('_')
    print("Parts:", parts)
    #print(str(parts[8+1].split('is')[1]))
    print("lambda:", parts[9+1].split('bda')[1])
    return {
        'latent_dim':    int(parts[0+1]),
        'num_layers_NN': int(parts[1+1]),
        'hidden_dim_NN': int(parts[2+1]),
        'noise_dim_dec': int(parts[3+1]),
        'hidden_dim_lm': int(parts[7+1]),
        'noise_dim_lm':  int(parts[5+1]),
        'encoder':       str(parts[8+1].split('is')[1]),
        'lambda':        float(parts[9+1].split('bda')[1]),
        'batch_norm':    str(parts[12+1].split('bnis')[1]),
        'alpha':         float(parts[10+1].split('lpha')[1]),
        'learn_rate':            float(parts[13+1].split('lr')[1])
    }
    


In [20]:
i = 1
while i < 1536:
    print(i)
    save_path = (
        f"{output_dir}_"
        f"{combo_dicts[i]['latent_dim']}_"
        f"{combo_dicts[i]['num_layers_NN']}_"
        f"{combo_dicts[i]['hidden_dim_NN']}_"
        f"{combo_dicts[i]['noise_dim_dec']}_1001_"
        f"{combo_dicts[i]['noise_dim_lm']}_2_"
        f"{combo_dicts[i]['hidden_dim_lm']}_"
        f"encoderislearnable_"
        #f"encoderis{combo_dicts[i]['encoder']}_"
        f"lambda{combo_dicts[i]['lambda']}_alpha{combo_dicts[i]['alpha']}_bs128_bnis{combo_dicts[i]['batch_norm']}_lr{combo_dicts[i]['learn_rate']}/log.txt"
    )
    df = load_and_validate(save_path, col_no)
    file_path = Path(save_path)
    
    train_losses = df[0]
    if np.isnan(train_losses).any():
        print("Contains NaN?", np.isnan(train_losses).any(), np.isnan(train_losses).sum())
    test_losses = df[1]
    if np.isnan(test_losses).any():
        print("Contains NaN?", np.isnan(test_losses).any(), np.isnan(test_losses).sum())

    #print(train_losses.shape, test_losses.shape)
    #print("filepath:", file_path.parent)
    params = parse_filename(file_path.parent)

    ###
    # train values
    ds['value'].loc[
        dict(
            latent_dim=params['latent_dim'],
            encoder=params['encoder'],
            hidden_dim_NN=params['hidden_dim_NN'],
            num_layers_NN=params['num_layers_NN'],
            noise_dim_dec=params['noise_dim_dec'],
            hidden_dim_lm=params['hidden_dim_lm'],
            noise_dim_lm=params['noise_dim_lm'],
            lamb=params['lambda'],
            alpha=params['alpha'],
            batch_norm=params['batch_norm'],
            learn_rate=params['learn_rate'],
            mode="train",
        )
    ] = train_losses
    
    # test values
    ds['value'].loc[
        dict(
            latent_dim=params['latent_dim'],
            encoder=params['encoder'],
            hidden_dim_NN=params['hidden_dim_NN'],
            num_layers_NN=params['num_layers_NN'],
            noise_dim_dec=params['noise_dim_dec'],
            hidden_dim_lm=params['hidden_dim_lm'],
            noise_dim_lm=params['noise_dim_lm'],
            lamb=params['lambda'],
            alpha=params['alpha'],
            batch_norm=params['batch_norm'],
            learn_rate=params['learn_rate'],
            mode="test",
        )
    ] = test_losses

    ###
    i += 1

1
_50_4_50_5_1001_20_2_50_encoderislearnable_lambda0.5_alpha1.0_bs128_bnisFalse_lr5e-05
Parts: ['', '50', '4', '50', '5', '1001', '20', '2', '50', 'encoderislearnable', 'lambda0.5', 'alpha1.0', 'bs128', 'bnisFalse', 'lr5e-05']
lambda: 0.5
2
_50_4_50_5_1001_20_2_50_encoderislearnable_lambda0.5_alpha1.5_bs128_bnisFalse_lr0.0001
Parts: ['', '50', '4', '50', '5', '1001', '20', '2', '50', 'encoderislearnable', 'lambda0.5', 'alpha1.5', 'bs128', 'bnisFalse', 'lr0.0001']
lambda: 0.5
3
_50_4_50_5_1001_20_2_50_encoderislearnable_lambda0.5_alpha1.5_bs128_bnisFalse_lr5e-05
Parts: ['', '50', '4', '50', '5', '1001', '20', '2', '50', 'encoderislearnable', 'lambda0.5', 'alpha1.5', 'bs128', 'bnisFalse', 'lr5e-05']
lambda: 0.5
4
_50_4_50_5_1001_20_2_50_encoderislearnable_lambda0.5_alpha1.0_bs128_bnisTrue_lr0.0001
Parts: ['', '50', '4', '50', '5', '1001', '20', '2', '50', 'encoderislearnable', 'lambda0.5', 'alpha1.0', 'bs128', 'bnisTrue', 'lr0.0001']
lambda: 0.5
5
_50_4_50_5_1001_20_2_50_encoderislearnab

ValueError: Validation failed for file: /work/fl53wumy-dpa_data/fl53wumy-llaae_data_new_22092025-1763346001/fl53wumy-llaae_data_new-1758244802/fl53wumy-llaae_data_new-1748049607/dpa_output/dpa_tuning5_starting13022026_v6_data/_100_6_100_20_1001_20_2_100_encoderislearnable_lambda0.5_alpha1.0_bs128_bnisFalse_lr5e-05/log.txt

In [62]:
ds.value.isnull().sum().item()


172800

In [61]:
192*100*16*9

2764800

In [64]:
ds.value

<xarray.DataArray 'value' (latent_dim: 2, encoder: 1, hidden_dim_NN: 2,
                           num_layers_NN: 2, noise_dim_dec: 3,
                           hidden_dim_lm: 2, noise_dim_lm: 2, lamb: 2,
                           epoch: 100, batch_norm: 2, alpha: 2, learn_rate: 2,
                           mode: 2, loss: 9)> Size: 22MB
array([[[[[[[[[[[[[[      nan,       nan,       nan, ...,
                          nan,       nan,       nan],
                   [      nan,       nan,       nan, ...,
                          nan,       nan,       nan]],

                  [[  88.828 ,  107.9777,   51.9258, ...,
                      46.8059,   64.8158,   36.0196],
                   [  52.5064,   91.7268,   78.4407, ...,
                      31.2445,   58.3335,   54.178 ]]],


                 [[[  90.932 ,   98.7808,   66.1158, ...,
                      38.0575,   60.6785,   45.2421],
                   [  42.7471,   82.9392,   80.3841, ...,
                      25.8789,   51.532 ,   51.3061]],

                  [[ 114.6983,  109.681 ,   51.1948, ...,
                      47.5354,   65.5114,   35.9519],
                   [  54.9026,   92.794 ,   75.7826, ...,
                      32.2654,   58.8041,   53.0774]]]],
...
                          nan,       nan,       nan],
                   [      nan,       nan,       nan, ...,
                          nan,       nan,       nan]],

                  [[      nan,       nan,       nan, ...,
                          nan,       nan,       nan],
                   [      nan,       nan,       nan, ...,
                          nan,       nan,       nan]]],


                 [[[      nan,       nan,       nan, ...,
                          nan,       nan,       nan],
                   [      nan,       nan,       nan, ...,
                          nan,       nan,       nan]],

                  [[      nan,       nan,       nan, ...,
                          nan,       nan,       nan],
                   [      nan,       nan,       nan, ...,
                          nan,       nan,       nan]]]]]]]]]]]]]],
      shape=(2, 1, 2, 2, 3, 2, 2, 2, 100, 2, 2, 2, 2, 9))
Coordinates: (12/14)
  * latent_dim     (latent_dim) int64 16B 50 100
  * encoder        (encoder) <U9 36B 'learnable'
  * hidden_dim_NN  (hidden_dim_NN) int64 16B 50 100
  * num_layers_NN  (num_layers_NN) int64 16B 4 6
  * noise_dim_dec  (noise_dim_dec) int64 24B 5 20 100
  * hidden_dim_lm  (hidden_dim_lm) int64 16B 50 100
    ...             ...
  * epoch          (epoch) int64 800B 1 2 3 4 5 6 7 8 ... 94 95 96 97 98 99 100
  * batch_norm     (batch_norm) <U5 40B 'False' 'True'
  * alpha          (alpha) float64 16B 1.0 1.5
  * learn_rate     (learn_rate) float64 16B 0.0001 5e-05
  * mode           (mode) <U5 40B 'train' 'test'
  * loss           (loss) <U10 360B 'Total loss' 'Total S1' ... 'LM S1' 'LM S2'

In [67]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# da: your xarray DataArray with dims including epoch, loss, mode, and all hyperparams
# Example: da = tuning_da

def plot_topk_lowest_loss_curves(da: xr.DataArray, k=5, mode_sel=None, loss_sel=None):
    # --- 1) Select mode and loss component ---
    if mode_sel is None:
        mode_sel = da.coords["mode"].values[0] if "mode" in da.dims else None
    if loss_sel is None:
        loss_sel = da.coords["loss"].values[0] if "loss" in da.dims else None

    sel = da
    if mode_sel is not None and "mode" in sel.dims:
        sel = sel.sel(mode=mode_sel)
    if loss_sel is not None and "loss" in sel.dims:
        sel = sel.sel(loss=loss_sel)

    # --- 2) Define "run dims" = all dims except epoch ---
    print(sel.dims)
    run_dims = [d for d in sel.dims if d != "epoch"]
    print(run_dims)
    # --- 3) For each run, compute the minimum loss achieved over epoch ---
    run_min = sel.min(dim="epoch", skipna=True)

    # Stack all hyperparam dims into one "run" index for easy sorting/selecting
    run_min_stacked = run_min.stack(run=run_dims)

    # Sort runs by achieved minimum loss (ascending) and take top-k
    order = np.argsort(run_min_stacked.values)
    topk_runs = run_min_stacked.isel(run=order[:k])

    # --- 4) Prepare curves stacked the same way and select the same top-k runs ---
    curves_stacked = sel.stack(run=run_dims)
    topk_curves = curves_stacked.sel(run=topk_runs["run"])

    # --- 5) Plot ---
    plt.figure()
    epoch = sel["epoch"].values

    for i in range(topk_curves.sizes["run"]):
        run_id = topk_curves["run"].values[i]  # this is a tuple of coordinates (MultiIndex)
        y = topk_curves.isel(run=i).values
        plt.plot(epoch, y, label=str(run_id))

    plt.xlabel("epoch")
    plt.ylabel(f"loss ({loss_sel})" if loss_sel is not None else "loss")
    title = f"Top {k} runs with smallest achieved loss"
    if mode_sel is not None:
        title += f" (mode={mode_sel})"
    plt.title(title)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    return topk_runs  # returns the run identifiers + their min losses


# ---- Usage ----
top5 = plot_topk_lowest_loss_curves(da, k=5, mode_sel="test", loss_sel="LM NRGY")
# If you don't know the coord names, you can omit mode_sel/loss_sel and it will take the first entry.


('epoch',)
[]


ValueError: Must pass non-zero number of levels/codes